# Qwen3 0.6B Sweep — Generation + Parsing

This launcher trains causal-LM adapters that generate `ham`/`spam` and evaluate by parsing the generated text.

Default configuration set: `K08`, `K11`, `K06`, `K09`, selected from the method-03 next-token ranking.

In [ ]:
from pathlib import Path
import json
import os
import signal
import subprocess
import sys

import pandas as pd

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "dataset").is_dir() and (candidate / "lora-fine-tuning").is_dir():
            return candidate
    raise RuntimeError("Could not find project root.")

PROJECT_ROOT = find_project_root()
METHOD_DIR = PROJECT_ROOT / "lora-fine-tuning" / "methods" / "02_causal_lm_generation_parsing"
SCRIPT = METHOD_DIR / "qwen3_0.6b_generation_parsing_sweep.py"
RESULTS_ROOT = METHOD_DIR / "results"

# Safe default: one tiny smoke run. For the A100 run, set SMOKE_TEST=False.
RUN_FULL_SWEEP = False
SMOKE_TEST = True
CONFIG_INDEXES = [8, 11, 6, 9]
SWEEP_ID = None

print(f"Project root: {PROJECT_ROOT}")
print(f"Script:       {SCRIPT}")
print(f"Results:      {RESULTS_ROOT}")


## Configurations

In [ ]:
configs = json.loads(
    subprocess.check_output(
        [sys.executable, str(SCRIPT), "list-configs", "--json"],
        cwd=PROJECT_ROOT,
        text=True,
    )
)
configs_df = pd.DataFrame(configs)
display(configs_df[[
    "index",
    "group",
    "max_seq_length",
    "learning_rate",
    "lora_r",
    "lora_alpha",
    "lora_dropout",
    "train_batch_size",
    "gradient_accumulation_steps",
    "effective_batch_size",
    "config_id",
]])


## Launch Sweep

In [ ]:
def terminate_process_tree(process: subprocess.Popen, timeout: float = 30.0) -> None:
    if process.poll() is not None:
        return
    if os.name == "nt":
        process.terminate()
    else:
        try:
            os.killpg(process.pid, signal.SIGINT)
        except ProcessLookupError:
            return
        except Exception:
            process.send_signal(signal.SIGINT)
    try:
        process.wait(timeout=timeout)
        return
    except subprocess.TimeoutExpired:
        pass
    if os.name == "nt":
        process.terminate()
    else:
        try:
            os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError:
            return
        except Exception:
            process.terminate()
    try:
        process.wait(timeout=10)
        return
    except subprocess.TimeoutExpired:
        pass
    if os.name == "nt":
        process.kill()
    else:
        try:
            os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError:
            return
        except Exception:
            process.kill()
    process.wait()


def latest_sweep_dir() -> Path | None:
    sweeps_root = RESULTS_ROOT / "sweeps"
    sweep_dirs = sorted(
        [path for path in sweeps_root.glob("qwen3_clm_generation_parsing_*") if path.is_dir()],
        key=lambda path: path.stat().st_mtime,
    )
    return sweep_dirs[-1] if sweep_dirs else None


def print_failure_summary() -> None:
    sweep_dir = latest_sweep_dir()
    if sweep_dir is None:
        print("No sweep directory found yet.")
        return
    summary_path = sweep_dir / "summary.csv"
    if not summary_path.exists():
        print(f"No summary.csv found in {sweep_dir}")
        return
    summary = pd.read_csv(summary_path)
    failed = summary[summary["status"].isin(["failed", "interrupted", "metrics_json_invalid"])]
    if failed.empty:
        print(f"No failed rows in {summary_path}")
        return
    display(failed[["config_index", "status", "error_type", "error_message", "run_dir"]])


def run_command(command: list[str]) -> None:
    print(" ".join(command))
    process = subprocess.Popen(command, cwd=PROJECT_ROOT, text=True, start_new_session=(os.name != "nt"))
    try:
        return_code = process.wait()
    except KeyboardInterrupt:
        terminate_process_tree(process)
        raise
    if return_code != 0:
        print_failure_summary()
        raise RuntimeError(f"Command exited with code {return_code}; see failed config details above.")


In [ ]:
command = [
    sys.executable,
    str(SCRIPT),
    "run-sweep",
    "--results-root",
    str(RESULTS_ROOT),
    "--resume",
]
if SWEEP_ID:
    command.extend(["--sweep-id", SWEEP_ID])
if not RUN_FULL_SWEEP:
    command.extend(["--config-index", ",".join(str(index) for index in CONFIG_INDEXES)])

if SMOKE_TEST:
    command.extend([
        "--allow-non-cuda",
        "--max-steps", "1",
        "--train-limit", "24",
        "--validation-limit", "8",
        "--test-limit", "8",
        "--cooldown-seconds", "0",
    ])
else:
    cuda_check = subprocess.run(
        [sys.executable, "-c", "import torch; raise SystemExit(0 if torch.cuda.is_available() else 1)"],
        cwd=PROJECT_ROOT,
    )
    if cuda_check.returncode != 0:
        raise RuntimeError("CUDA is not available. Keep SMOKE_TEST=True for local checks or run this on the A100 box.")

run_command(command)


## Summary

In [ ]:
sweep_dir = latest_sweep_dir()
if sweep_dir is None:
    raise FileNotFoundError(f"No sweeps found under {RESULTS_ROOT / 'sweeps'}")
summary_path = sweep_dir / "summary.csv"
print(f"Loaded: {summary_path}")
summary = pd.read_csv(summary_path)
leaderboard = summary.sort_values(["status", "validation_f1", "test_f1"], ascending=[True, False, False])
display(leaderboard)


## Rebuild Summary

In [ ]:
sweep_dir = latest_sweep_dir()
if sweep_dir is None:
    raise FileNotFoundError(f"No sweeps found under {RESULTS_ROOT / 'sweeps'}")
subprocess.run(
    [sys.executable, str(SCRIPT), "summarize", "--sweep-dir", str(sweep_dir)],
    cwd=PROJECT_ROOT,
    check=True,
)
summary = pd.read_csv(sweep_dir / "summary.csv")
summary.sort_values(["status", "validation_f1", "test_f1"], ascending=[True, False, False])
